# 第3周：网络

> **学习目标**：理解 TCP/IP 协议栈、掌握 ss/tcpdump 调试工具、DNS 解析链路、iptables 防火墙、容器网络原理（namespace/veth/bridge）、Docker 与 iptables 的坑

---

## 开篇：网络是看不见的基础设施

你在浏览器输入 https://www.baidu.com，回车，页面出来了。这背后发生了什么？

1. DNS 查询：www.baidu.com → IP 地址
2. TCP 三次握手：SYN → SYN-ACK → ACK
3. TLS 握手（HTTPS）：协商加密参数
4. HTTP 请求：GET / → 返回 HTML

这周我们理解网络出问题时如何排查——DNS 解析慢？TCP 丢包？防火墙拦截？


---

## Day 15：TCP/IP 协议栈

### 四层模型

应用层：HTTP / DNS / SSH（你的程序在这里）
传输层：TCP（可靠）/ UDP（快速）
网络层：IP（寻址、路由）
链路层：Ethernet（MAC 地址、帧）

数据封装：应用数据 → HTTP 头 → TCP 头 → IP 头 → 以太帧头。

### TCP 三次握手

客户端 → SYN → 服务端 → SYN-ACK → 客户端 → ACK → 连接建立

### TCP 四次挥手

主动方 → FIN → 被动方 → ACK → 被动方 → FIN → 主动方 → ACK → TIME_WAIT

TIME_WAIT 存在的原因：确保最后一个 ACK 不会丢。


In [ ]:
# 用 ss 观察 TCP 状态
! echo "所有 TCP 连接的状态："
! ss -tan | head -15
! echo ""
! echo "统计各状态连接数："
! ss -tan | awk 'NR>1{print $1}' | sort | uniq -c | sort -rn
! echo ""
! echo "当前监听端口："
! ss -tlnp 2>/dev/null | awk 'NR>1{printf "%-20s %s\n", $4, $NF}' || ss -tln | awk 'NR>1{print $4}'
! echo ""
! echo "只看 TIME_WAIT："
! ss -tan state time-wait | head -5
! echo ""
! echo "统计摘要："
! ss -s

In [ ]:
# 启动 HTTP 服务观察 TCP
! python3 -m http.server 8765 --bind 127.0.0.1 &
! sleep 1
! echo "确认监听："
! ss -tlnp | grep 8765 || echo "端口未找"
! curl -s -o /dev/null -w "HTTP: %{http_code}\n" http://127.0.0.1:8765/
! kill %1 2>/dev/null; wait 2>/dev/null
! echo "已清理"

---

## Day 16：ss / tcpdump — 网络调试双雄

ss 是 netstat 的现代替代品。常用组合：`ss -tlnp`（列出 TCP 监听端口及对应进程）。

tcpdump 核心语法：
```
tcpdump -i any port 80        # 抓 HTTP
tcpdump -i eth0 host x.x.x.x  # 抓特定主机
tcpdump -w file.pcap          # 保存到文件
tcpdump -r file.pcap          # 读取文件
tcpdump -A                    # ASCII 显示包内容
```


In [ ]:
# ss 高级用法
! echo "连接按对方 IP 统计："
! ss -tan | awk 'NR>1{print $5}' | cut -d: -f1 | sort | uniq -c | sort -rn | head -5
! echo ""
! cat << 'DEMO'
# tcpdump 演示（需要 root）：
# 终端1：python3 -m http.server 8888 &
# 终端2：sudo tcpdump -i lo -A port 8888
# 终端3：curl http://localhost:8888/
#
# 输出会看到 TCP 三次握手：
# Flags [S]  = SYN
# Flags [S.] = SYN-ACK
# Flags [.]  = ACK
DEMO

---

## Day 17：DNS 解析链路

DNS 查询流程：浏览器 → 缓存 → /etc/hosts → 解析器 → 根域名 → 顶级域 → 权威服务器

关键文件：
- `/etc/resolv.conf`：DNS 服务器地址
- `/etc/hosts`：本地静态解析（优先级最高）
- `/etc/nsswitch.conf`：控制解析顺序（files dns）


In [ ]:
# dig — DNS 查询瑞士军刀
! echo "当前 DNS 配置："
! cat /etc/resolv.conf 2>/dev/null || echo "无法读取"
! echo ""
! dig +short www.baidu.com 2>/dev/null || echo "dig 未安装（apt install dnsutils）"
! echo ""
! if command -v dig &>/dev/null; then
!     dig www.baidu.com 2>/dev/null | grep -E "^;; " | head -6
! fi
! echo ""
! echo "查询类型："
! echo "  dig A example.com          # IPv4 地址"
! echo "  dig MX example.com         # 邮件服务器"
! echo "  dig NS example.com         # 域名服务器"
! echo "  dig CNAME www.example.com  # 别名"

In [ ]:
# 高级 DNS 查询
! echo "=== 追踪完整解析链路 ==="
! dig +trace www.baidu.com 2>/dev/null | head -20 || echo "+trace 需要网络"
! echo ""
! echo "=== 不同 DNS 服务器对比 ==="
! time dig @8.8.8.8 +short www.baidu.com 2>/dev/null || echo "无法连接 8.8.8.8"
! echo ""
! echo "=== 反向 DNS ==="
! dig -x 8.8.8.8 +short 2>/dev/null || echo "反向查询失败"
! echo ""
! cat /etc/hosts 2>/dev/null | grep -v '^#' | grep -v '^$'
! echo ""
! echo "DNS 性能排查：time dig example.com 超过 500ms 说明 DNS 太慢"

---

## Day 18：iptables 防火墙

### 表与链

| 表 | 主要链 | 用途 |
|------|--------|------|
| filter | INPUT, OUTPUT, FORWARD | 包过滤（默认表） |
| nat | PREROUTING, POSTROUTING, OUTPUT | 地址转换 |
| mangle | PREROUTING, INPUT, FORWARD, OUTPUT, POSTROUTING | 修改包头部 |

### 规则格式

`iptables -A <链> -s <源IP> -p <协议> --dport <端口> -j <动作>`

动作：ACCEPT（允许）/ DROP（静默丢弃）/ REJECT（拒绝）/ LOG（记录）

### 核心规则模板

```bash
iptables -P INPUT DROP
iptables -P FORWARD DROP
iptables -P OUTPUT ACCEPT
iptables -A INPUT -i lo -j ACCEPT
iptables -A INPUT -m conntrack --ctstate ESTABLISHED,RELATED -j ACCEPT
iptables -A INPUT -p tcp --dport 22 -s 192.168.0.0/24 -j ACCEPT
iptables -A INPUT -p tcp --dport 80 -j ACCEPT
```


In [ ]:
# iptables 规则查看
! echo "filter 表规则："
! sudo iptables -L -n -v 2>/dev/null | head -15 || echo "查看规则需要 root"
! echo ""
! echo "nat 表规则："
! sudo iptables -t nat -L -n -v 2>/dev/null | head -15 || echo "无权限或未启用"
! echo ""
! cat << 'NAT_DEMO'
# Docker 端口映射的底层原理：
# docker run -p 8080:80 nginx 等价于：
#
# 1. DNAT：宿主机 8080 → 容器 172.17.0.2:80
# iptables -t nat -A PREROUTING -p tcp --dport 8080 \
#   -j DNAT --to-destination 172.17.0.2:80
#
# 2. 允许 FORWARD
# iptables -A FORWARD -d 172.17.0.2 -p tcp --dport 80 -j ACCEPT
#
# 3. SNAT：容器回包时源地址改为宿主机 IP
# iptables -t nat -A POSTROUTING -s 172.17.0.2 \
#   -p tcp --sport 80 -j MASQUERADE
NAT_DEMO

---

## Day 19：容器网络原理

### 三大技术支柱

| 技术 | 作用 |
|------|------|
| Network Namespace | 网络栈隔离（独立网卡、路由表） |
| veth pair | 虚拟网线，一头连容器一头连宿主机 |
| Linux Bridge | 虚拟交换机（docker0）连接所有 veth |

Docker 网络模型：容器 eth0 → veth → docker0 网桥 → 宿主机 eth0 → 互联网


In [ ]:
# 手动创建 network namespace
! cat << 'DEMO'
# 创建两个网络命名空间
sudo ip netns add ns1
sudo ip netns add ns2

# 创建 veth pair（虚拟网线）
sudo ip link add veth1 type veth peer name veth2

# 把 veth 移到命名空间
sudo ip link set veth1 netns ns1
sudo ip link set veth2 netns ns2

# 配置 IP
sudo ip netns exec ns1 ip addr add 10.0.0.1/24 dev veth1
sudo ip netns exec ns1 ip link set veth1 up
sudo ip netns exec ns2 ip addr add 10.0.0.2/24 dev veth2
sudo ip netns exec ns2 ip link set veth2 up

# 测试连通
sudo ip netns exec ns1 ping 10.0.0.2 -c 3

# 清理
sudo ip netns delete ns1
sudo ip netns delete ns2
DEMO
! echo ""
! echo "现有 network namespace："
! sudo ip netns list 2>/dev/null || echo "无命名空间"
! echo ""
! echo "进入容器网络空间（nsenter）："
! echo "  CONTAINER_ID=\\$(docker ps -q | head -1)"
! echo "  PID=\\$(docker inspect \$CONTAINER_ID -f '{{.State.Pid}}')"
! echo "  sudo nsenter -t \$PID -n ip addr"
! echo "  sudo nsenter -t \$PID -n ss -tln"

---

## Day 20：防火墙与 Docker 的坑

### 问题 1：手动 iptables 规则被 Docker 绕过

Docker 的 -p 在 nat 表 PREROUTING 做 DNAT，不经 filter 表的 INPUT 链。
即使 `iptables -P INPUT DROP`，容器端口仍然暴露。

**解决**：在 DOCKER-USER 链添加规则：
```
iptables -I DOCKER-USER -i eth0 -p tcp --dport 80 -j DROP
```

### 问题 2：外网也能访问

```bash
docker run -p 127.0.0.1:8080:80 nginx  # 只本机能访问
docker run -p 8080:80 nginx            # 所有 IP 都能访问
```

### 问题 3：UFW 与 Docker 冲突

UFW 管理 filter 表，Docker 操作 nat 表，不在同一层面。
使用 `--iptables=false` 禁用 Docker 自动修改。


In [ ]:
# 验证 Docker 端口映射
! echo "Docker 在 nat 表的规则："
! sudo iptables -t nat -L -n 2>/dev/null | grep -E "DOCKER|8080|80" | head -10 || echo "无 Docker 规则"
! echo ""
! echo "DOCKER-USER 链："
! sudo iptables -L DOCKER-USER -n 2>/dev/null || echo "无 DOCKER-USER 链"
! echo ""
! echo "安全建议："
! echo "  1. 只本机访问：-p 127.0.0.1:8080:80"
! echo "  2. 限制外部访问：DOCKER-USER 链"
! echo "  3. 禁用 Docker 改 iptables：--iptables=false"

---

## Day 21：第三周综合练习

### 配置带防火墙的 Docker 主机

**防火墙基础**：
- INPUT DROP / FORWARD DROP / OUTPUT ACCEPT
- 允许 lo 和 ESTABLISHED,RELATED
- SSH（22）仅限内网
- HTTP（80）和 HTTPS（443）对外开放
- SSH 防暴力破解（每分钟 3 次限制）

**Docker 集成**：
- Nginx 容器映射 80 端口（对外访问）
- 另一个容器只监听 127.0.0.1:8081（仅本机）
- 用 DOCKER-USER 链限制容器端口

**网络排查**：
- tcpdump 抓取 HTTP 请求-响应
- nsenter 进入容器 network namespace
- dig +trace 追踪 DNS 解析路径


In [ ]:
# 防火墙配置脚本参考
! cat << 'FW'
#!/bin/bash
# 注意：逐条理解后再执行！

echo "=== 默认策略 ==="
iptables -P INPUT DROP
iptables -P FORWARD DROP
iptables -P OUTPUT ACCEPT

echo "=== 允许回环和已建立连接 ==="
iptables -A INPUT -i lo -j ACCEPT
iptables -A INPUT -m conntrack --ctstate ESTABLISHED,RELATED -j ACCEPT

echo "=== 开放端口 ==="
iptables -A INPUT -p tcp --dport 22 -s 192.168.0.0/24 -j ACCEPT
iptables -A INPUT -p tcp --dport 80 -j ACCEPT
iptables -A INPUT -p tcp --dport 443 -j ACCEPT

echo "=== SSH 防暴力破解 ==="
iptables -A INPUT -p tcp --dport 22 -m state --state NEW \
    -m recent --set --name SSH
iptables -A INPUT -p tcp --dport 22 -m state --state NEW \
    -m recent --update --seconds 60 --hitcount 3 --name SSH -j DROP

echo "=== Docker 保护 ==="
iptables -I DOCKER-USER -i eth0 -p tcp --dport 80 -j DROP 2>/dev/null || true
FW

---

## 第3周总结

| 概念 | 一句话 | 关键命令 |
|------|--------|----------|
| TCP/IP 四层模型 | 应用→传输→网络→链路 | |
| 三次握手 | SYN→SYN-ACK→ACK | |
| TIME_WAIT | 主动关闭方等 2MSL | `ss -tan state time-wait` |
| ss | socket 状态查询 | `ss -tlnp` |
| tcpdump | 抓包器 | `tcpdump -i any port 80 -A` |
| dig | DNS 查询 | `dig +trace example.com` |
| iptables | 防火墙 | `iptables -L -n -v` |
| DNAT | 端口映射（Docker -p 实现） | |
| Network Namespace | 网络隔离 | `ip netns add` |
| veth pair | 虚拟网线 | `ip link add ... type veth` |
| DOCKER-USER | Docker 防火墙正确用法 | |

### 肌肉记忆

```bash
ss -tlnp                        # 查看监听端口
ss -tan state time-wait         # 只看 TIME_WAIT
sudo tcpdump -i any port 80    # 抓 HTTP 包
dig +trace example.com          # DNS 全链路追踪
sudo iptables -L -n -v          # 查看防火墙
sudo ip netns add ns1           # 创建网络命名空间
```
